In [1]:
import tkinter as tk
from tkinter import ttk, messagebox
from tkinter.scrolledtext import ScrolledText
import joblib
import pandas as pd
import os

MODEL = joblib.load("Salary_best_lightgbm_regression_model.pkl")


department_map = {
    "Engineering": 0,
    "Finance": 1,
    "HR": 2,
    "Marketing": 3,
    "Product": 4,
    "Sales": 5
}

job_title_map = {
    "Analyst": 0,
    "Engineer": 1,
    "Executive": 2,
    "Intern": 3,
    "Manager": 4
}

education_map = {
    "Bachelor": 0,
    "Master": 1,
    "PhD": 2
}

location_map = {
    "Austin": 0,
    "Chicago": 1,
    "New York": 2,
    "San Francisco": 3,
    "Seattle": 4
}


root = tk.Tk()

root.title("Employee Salary Prediction Prototype")

root.geometry("950x760")

root.configure(bg="#EAF4FC")



title = tk.Label(
    root,
    text="Employee Salary Prediction Prototype",
    bg="#1565C0",
    fg="white",
    font=("Segoe UI",20,"bold"),
    pady=10
)

title.pack(fill="x")



frm = tk.Frame(
    root,
    bg="#EAF4FC"
)

frm.pack(pady=15)

labels = [

    ("Age","entry"),

    ("Department","combo"),

    ("Job_Title","combo"),

    ("Experience_Years","entry"),

    ("Education_Level","combo"),

    ("Location","combo")

]

widgets = {}

for r,(name,kind) in enumerate(labels):

    tk.Label(

        frm,

        text=name.replace("_"," "),

        bg="#EAF4FC",

        font=("Segoe UI",11,"bold")

    ).grid(row=r,column=0,sticky="w",padx=10,pady=8)

    if kind=="entry":

        entry = ttk.Entry(frm,width=35)

        entry.grid(row=r,column=1,padx=10,pady=8)

        widgets[name]=entry

    else:

        if name=="Department":

            values=list(department_map.keys())

        elif name=="Job_Title":

            values=list(job_title_map.keys())

        elif name=="Education_Level":

            values=list(education_map.keys())

        else:

            values=list(location_map.keys())

        combo=ttk.Combobox(

            frm,

            values=values,

            width=33,

            state="readonly"

        )

        combo.current(0)

        combo.grid(

            row=r,

            column=1,

            padx=10,

            pady=8

        )

        widgets[name]=combo



result = tk.StringVar()

result.set("Predicted Salary : -")

output = tk.Label(

    root,

    textvariable=result,

    bg="#D5F5E3",

    fg="#145A32",

    font=("Segoe UI",18,"bold"),

    relief="ridge",

    bd=2,

    pady=15

)

output.pack(

    fill="x",

    padx=30,

    pady=10

)


def predict():

    try:

        age = float(widgets["Age"].get())

        experience = float(

            widgets["Experience_Years"].get()

        )

        if age < 18 or age > 65:

            messagebox.showwarning(

                "Invalid Age",

                "Age should be between 18 and 65."

            )

            return

        if experience < 0:

            messagebox.showwarning(

                "Invalid Experience",

                "Experience cannot be negative."

            )

            return

        if experience > (age-18):

            messagebox.showwarning(

                "Invalid Experience",

                "Experience Years cannot exceed Age - 18."

            )

            return

        input_df = pd.DataFrame([{

            "Age": age,

            "Department":

            department_map[

                widgets["Department"].get()

            ],

            "Job_Title":

            job_title_map[

                widgets["Job_Title"].get()

            ],

            "Experience_Years":

            experience,

            "Education_Level":

            education_map[

                widgets["Education_Level"].get()

            ],

            "Location":

            location_map[

                widgets["Location"].get()

            ]

        }])

        print("\nInput Sent To Model")

        print(input_df)

        # IMPORTANT:
        # No feature scaling because the model
        # was trained on X_train (original features).

        prediction = MODEL.predict(input_df)[0]

        result.set(

            f"Predicted Salary : ₹ {prediction:,.2f}"

        )

    except Exception as e:

        messagebox.showerror(

            "Prediction Error",

            str(e)

        )


def reset():

    for key, widget in widgets.items():

        if isinstance(widget, ttk.Entry):

            widget.delete(0, tk.END)

        else:

            widget.current(0)

    result.set("Predicted Salary : -")



button_frame = tk.Frame(
    root,
    bg="#EAF4FC"
)

button_frame.pack(pady=10)

predict_button = tk.Button(

    button_frame,

    text="Predict Salary",

    bg="#2E7D32",

    fg="white",

    font=("Segoe UI",11,"bold"),

    width=18,

    command=predict

)

predict_button.grid(
    row=0,
    column=0,
    padx=10
)

reset_button = tk.Button(

    button_frame,

    text="Reset",

    bg="#FB8C00",

    fg="white",

    font=("Segoe UI",11,"bold"),

    width=18,

    command=reset

)

reset_button.grid(
    row=0,
    column=1,
    padx=10
)



evaluation_frame = tk.LabelFrame(

    root,

    text="Human Evaluation",

    bg="#EAF4FC",

    font=("Segoe UI",12,"bold")

)

evaluation_frame.pack(

    fill="both",

    expand=True,

    padx=20,

    pady=20

)



tk.Label(

    evaluation_frame,

    text="Overall Rating",

    bg="#EAF4FC",

    font=("Segoe UI",10,"bold")

).grid(

    row=0,

    column=0,

    padx=10,

    pady=8,

    sticky="w"

)

rating = ttk.Combobox(

    evaluation_frame,

    values=[1,2,3,4,5],

    width=12,

    state="readonly"

)

rating.current(4)

rating.grid(

    row=0,

    column=1,

    padx=10,

    pady=8

)



recommend = tk.StringVar()

recommend.set("Yes")

tk.Label(

    evaluation_frame,

    text="Would you recommend this prototype?",

    bg="#EAF4FC",

    font=("Segoe UI",10,"bold")

).grid(

    row=1,

    column=0,

    padx=10,

    pady=8,

    sticky="w"

)

recommend_combo = ttk.Combobox(

    evaluation_frame,

    textvariable=recommend,

    values=["Yes","No"],

    width=12,

    state="readonly"

)

recommend_combo.grid(

    row=1,

    column=1,

    padx=10,

    pady=8

)



tk.Label(

    evaluation_frame,

    text="Comments",

    bg="#EAF4FC",

    font=("Segoe UI",10,"bold")

).grid(

    row=2,

    column=0,

    padx=10,

    pady=5,

    sticky="nw"

)

comments = ScrolledText(

    evaluation_frame,

    width=60,

    height=3,

    font=("Segoe UI",10)

)

comments.grid(

    row=2,

    column=1,

    padx=10,

    pady=8

)


def save_feedback():

    feedback = pd.DataFrame({

        "Rating":[rating.get()],

        "Recommendation":[recommend.get()],

        "Comments":[comments.get("1.0",tk.END).strip()]

    })

    filename = "Salary_Human_Evaluation.csv"

    if os.path.exists(filename):

        feedback.to_csv(

            filename,

            mode="a",

            header=False,

            index=False

        )

    else:

        feedback.to_csv(

            filename,

            index=False

        )

    messagebox.showinfo(

        "Success",

        "Human evaluation feedback saved successfully."

    )

    comments.delete(

        "1.0",

        tk.END

    )

submit_button = tk.Button(

    evaluation_frame,

    text="Submit Feedback",

    bg="#1976D2",

    fg="white",

    font=("Segoe UI",11,"bold"),

    width=18,

    command=save_feedback

)

submit_button.grid(

    row=3,

    column=1,

    pady=15,

    sticky="e"

)



root.mainloop()